# NB41: Feature Selection EDA Validation

**Hipotez:** Danışmanın EDA'sından çıkan feature-depth bulgusu (top-200 feature > all feature, train-dağılımı F1: 0.894 vs 0.879) bizim doğrulanmış %80/20 bootstrap (ters dağılım) protokolümüzde de geçerli mi?

**Bağlam:** Danışman XGBoost feature importance'ı kullanarak MASTER'da top-200 feature'ın daha iyi performans verdiğini gösterdi — kuyruk feature'lar gürültü. Ancak bu doğrulandı sadece train-dağılımında (stratified hold-out). Bizim gerçek soru:
- Danışmanın train-dağılımı kazancı, ters dağılımda (%80/20 bootstrap) da geçerli mi?
- Bu feature-seçimi size 288–440 feature'dan daha iyi bir modelle geçeceğimizi mi sağlayacak?

**Referans:** NB39'u taklit ediyor (reversed-distribution protokolü, %80/20 bootstrap eval, floor-karşılaştırması).

## Deney Tasarımı

Her panel (MASTER, KANSER, PAH) için:
1. Stratified %80/20 split (SEED=42), sabit+özdeş sütun temizliği, M3 preprocessing
2. **XGBoost feature ranking:** train üzerinde feature importance (gain) hesapla
3. **Feature subset sweep:** k ∈ {10, 25, 50, 100, 200, all}
4. **Her k için model:** BalancedBaggingClassifier (10 estimator, LGBMClassifier base)
5. **Değerlendirme:** NB39 protokolü (%80/20 bootstrap, N=50, 95% CI) → F1, MCC, precision, recall
6. **Floor referansı:** prev≈0.20 için floor = 2×0.20/(1+0.20) ≈ 0.333

**Ana Soru:** Her panelde "top-N en iyi k" vs "all feature" Boot-F1 farkı. Danışmanın kazancı korunuyor mu?

| Panel  | Dağılım | n_rows | pos  | neg  | pos_rate |
|--------|---------|--------|------|------|----------|
| MASTER | ~73:27  | 2931   | 2149 | 782  | 0.734    |
| KANSER | ~69:31  | 388    | 268  | 120  | 0.691    |
| PAH    | ~17:83  | 372    | 62   | 310  | 0.167    |

CFTR (n=111, neg=21) atlanıyor — bootstrap CI [0–1] anlamsız.

In [1]:
# Cell 1: Imports ve Setup
import sys, os, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
    PROJECT_ROOT = os.path.dirname(os.getcwd())
if not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
    PROJECT_ROOT = os.getcwd()
    while PROJECT_ROOT != '/' and not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

from config import SEED, TEST_SIZE, PROJECT_ROOT, REPORTS_DIR
from src.columns_real import (
    ID_COL, TARGET_COL, NON_FEATURE_COLS,
    get_constant_cols, get_duplicate_col_pairs, get_missing_mask_col_name
)
from src.metrics import optimize_threshold, compute_all_metrics

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, matthews_corrcoef
from sklearn.metrics import precision_score, recall_score
from imblearn.ensemble import BalancedBaggingClassifier

import lightgbm as lgb
import xgboost as xgb
from fpdf import FPDF

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v24_feature_selection')
os.makedirs(RESULTS_DIR, exist_ok=True)

N_BOOTSTRAP = 50
N_FOLDS = 5
FLOOR_F1_THRESHOLD = 2 * 0.20 / (1 + 0.20)  # ~0.333 for prev=0.20

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'SEED: {SEED}, TEST_SIZE: {TEST_SIZE}')
print(f'Results: {RESULTS_DIR}')
print(f'Floor F1 threshold (prev=0.20): {FLOOR_F1_THRESHOLD:.4f}')

PROJECT_ROOT: /Users/tefe/teknofest_model/teknofest_model
SEED: 42, TEST_SIZE: 0.2
Results: /Users/tefe/teknofest_model/teknofest_model/results/v24_feature_selection
Floor F1 threshold (prev=0.20): 0.3333


In [2]:
# Cell 2: Veri Yükleme ve Sütun Temizliği (Template)
def load_and_clean_panel(panel_name):
    """
    Bir panel için veri yükle, split et, sabit+özdeş sütun temizle.
    Returns: X_train_raw, X_test_raw, y_train, y_test, feature_cols, const_cols, dup_col_pairs
    """
    csv_path = os.path.join(PROJECT_ROOT, f'data/real_data/YARISMA_TRAIN_{panel_name}.csv')
    df = pd.read_csv(csv_path)
    
    print(f'{panel_name} yüklendi: {df.shape}')
    print(f'Label dağılımı:')
    print(df[TARGET_COL].value_counts())
    
    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    X = df[feature_cols].copy()
    y = df[TARGET_COL].copy()
    
    # Stratified split
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    
    # Sabit ve özdeş sütun temizliği (train üzerinde tespit)
    const_cols = get_constant_cols(X_train_raw)
    dup_col_pairs = get_duplicate_col_pairs(X_train_raw)
    dup_cols = set(c2 for c1, c2 in dup_col_pairs)
    
    drop_cols = list(set(const_cols + list(dup_cols)))
    if 'CAT_6' in X_train_raw.columns and 'CAT_6' not in drop_cols:
        drop_cols.append('CAT_6')
    
    feature_cols = [c for c in feature_cols if c not in drop_cols]
    X_train_raw = X_train_raw[feature_cols].copy()
    X_test_raw = X_test_raw[feature_cols].copy()
    
    print(f'Train: {X_train_raw.shape} (pos={y_train.sum()}, neg={(y_train==0).sum()})')
    print(f'Test:  {X_test_raw.shape} (pos={y_test.sum()}, neg={(y_test==0).sum()})')
    print(f'Sabit={len(const_cols)}, Özdeş çift={len(dup_col_pairs)}, Toplam drop={len(drop_cols)}, Kalan={len(feature_cols)}')
    print()
    
    return X_train_raw, X_test_raw, y_train, y_test, feature_cols, const_cols, dup_col_pairs

# Test with MASTER
X_train_m, X_test_m, y_train_m, y_test_m, feat_m, const_m, dup_m = load_and_clean_panel('MASTER')

MASTER yüklendi: (2931, 353)
Label dağılımı:
Label
1    2149
0     782
Name: count, dtype: int64
Train: (2344, 287) (pos=1719, neg=625)
Test:  (587, 287) (pos=430, neg=157)
Sabit=57, Özdeş çift=583, Toplam drop=64, Kalan=287



In [3]:
# Cell 3: Preprocessing Pipeline (M3 missing + imputation)
def preprocess_scenario(X_tr, X_te):
    """
    M3 missing + LabelEncode + medyan imputation. Train'e fit, test'e transform.
    """
    X_tr = X_tr.copy()
    X_te = X_te.copy()

    # M3: is_missing flags (>%50 missing sütunlar)
    missing_threshold = 0.50
    for col in X_tr.columns:
        miss_ratio = X_tr[col].isnull().sum() / len(X_tr)
        if miss_ratio > missing_threshold:
            mask_col = get_missing_mask_col_name(col)
            X_tr[mask_col] = X_tr[col].isnull().astype(int)
            X_te[mask_col] = X_te[col].isnull().astype(int)

    # Medyan imputation (numeric)
    numeric_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    imputer = SimpleImputer(strategy='median')
    if numeric_cols:
        X_tr[numeric_cols] = imputer.fit_transform(X_tr[numeric_cols])
        X_te[numeric_cols] = imputer.transform(X_te[numeric_cols])

    # LabelEncode kategorikler
    cat_like = [c for c in X_tr.columns if X_tr[c].dtype == object or str(X_tr[c].dtype) == 'category']
    for col in cat_like:
        le = LabelEncoder()
        tr_vals = X_tr[col].astype('object').where(X_tr[col].notna(), '__NA__').astype(str)
        fit_vals = pd.concat([tr_vals, pd.Series(['__NA__'])], ignore_index=True)
        le.fit(fit_vals)
        X_tr[col] = le.transform(tr_vals)
        known = set(le.classes_)
        te_vals = X_te[col].astype('object').where(X_te[col].notna(), '__NA__').astype(str)
        te_vals = te_vals.map(lambda v, k=known: v if v in k else '__NA__')
        X_te[col] = le.transform(te_vals)

    # Kalan NaN'ları sıfırla
    for col in X_tr.columns:
        tr_num = pd.to_numeric(X_tr[col], errors='coerce')
        te_num = pd.to_numeric(X_te[col], errors='coerce')
        fill = tr_num.median()
        if pd.isna(fill):
            fill = 0.0
        X_tr[col] = tr_num.fillna(fill).astype(float)
        X_te[col] = te_num.fillna(fill).astype(float)

    return X_tr, X_te, imputer

print('Preprocessing pipeline hazır.')

Preprocessing pipeline hazır.


In [4]:
# Cell 4: %80/20 Bootstrap Degerlendirme Fonksiyonları
def optimize_threshold_8020(y_true, y_prob, n_bootstrap=N_BOOTSTRAP, target_pos_rate=0.20):
    """%80/20 dağılımda F1-max threshold seç."""
    best_thrs_f1, best_thrs_mcc = [], []
    rng = np.random.RandomState(SEED)
    idx_pos = np.where(y_true == 1)[0]
    idx_neg = np.where(y_true == 0)[0]
    for _ in range(n_bootstrap):
        n_neg = len(idx_neg)
        n_pos_target = max(1, int(n_neg * target_pos_rate / (1 - target_pos_rate)))
        if n_pos_target > len(idx_pos):
            n_pos_target = len(idx_pos)
        sel_pos = rng.choice(idx_pos, size=n_pos_target, replace=True)
        sel_neg = rng.choice(idx_neg, size=n_neg, replace=True)
        sel = np.concatenate([sel_pos, sel_neg])
        y_sel = y_true.values[sel] if hasattr(y_true, 'values') else y_true[sel]
        p_sel = y_prob[sel]
        best_f1, best_t_f1 = 0, 0.5
        best_mcc, best_t_mcc = -1, 0.5
        for thr in np.arange(0.10, 0.90, 0.01):
            preds = (p_sel >= thr).astype(int)
            f1 = f1_score(y_sel, preds, zero_division=0)
            mcc = matthews_corrcoef(y_sel, preds)
            if f1 > best_f1:
                best_f1, best_t_f1 = f1, thr
            if mcc > best_mcc:
                best_mcc, best_t_mcc = mcc, thr
        best_thrs_f1.append(best_t_f1)
        best_thrs_mcc.append(best_t_mcc)
    return np.median(best_thrs_f1), np.median(best_thrs_mcc)


def bootstrap_8020_eval(y_true, y_prob, threshold, n_bootstrap=N_BOOTSTRAP, target_pos_rate=0.20):
    """%80/20 dağılımda metrikler hesapla (N=50 bootstrap sample)."""
    rng = np.random.RandomState(SEED + 1)
    idx_pos = np.where(y_true == 1)[0]
    idx_neg = np.where(y_true == 0)[0]
    f1s, mccs, precs, recs = [], [], [], []
    for _ in range(n_bootstrap):
        n_neg = len(idx_neg)
        n_pos_target = max(1, int(n_neg * target_pos_rate / (1 - target_pos_rate)))
        if n_pos_target > len(idx_pos):
            n_pos_target = len(idx_pos)
        sel_pos = rng.choice(idx_pos, size=n_pos_target, replace=True)
        sel_neg = rng.choice(idx_neg, size=n_neg, replace=True)
        sel = np.concatenate([sel_pos, sel_neg])
        y_sel = y_true.values[sel] if hasattr(y_true, 'values') else y_true[sel]
        p_sel = y_prob[sel]
        preds = (p_sel >= threshold).astype(int)
        f1s.append(f1_score(y_sel, preds, zero_division=0))
        mccs.append(matthews_corrcoef(y_sel, preds))
        precs.append(precision_score(y_sel, preds, zero_division=0))
        recs.append(recall_score(y_sel, preds, zero_division=0))
    return {
        'f1_mean': float(np.mean(f1s)), 'f1_std': float(np.std(f1s)),
        'f1_ci_lo': float(np.percentile(f1s, 2.5)), 'f1_ci_hi': float(np.percentile(f1s, 97.5)),
        'mcc_mean': float(np.mean(mccs)), 'mcc_std': float(np.std(mccs)),
        'prec_mean': float(np.mean(precs)), 'rec_mean': float(np.mean(recs)),
    }

print('Bootstrap degerlendirme fonksiyonları hazır.')

Bootstrap degerlendirme fonksiyonları hazır.


In [5]:
# Cell 5: XGBoost Feature Importance Ranklamasi
def get_xgb_feature_importance_ranking(X_train, y_train, seed=SEED):
    """
    Train setinde XGBoost eğit, feature importance (gain) ile sırala.
    Returns: feature_names_sorted (gain'e göre azalan sıra)
    """
    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        verbose=0,
        eval_metric='logloss'
    )
    model.fit(X_train, y_train)
    
    # Feature importance (gain) sırasına göre
    importance_df = pd.DataFrame({
        'feature': X_train.columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    return importance_df

print('Feature importance ranking fonksiyonu hazır.')

Feature importance ranking fonksiyonu hazır.


In [6]:
# Cell 6: Feature Subset Egitim ve Degerlendirme
def train_and_eval_feature_subset(panel_name, X_train_raw, X_test_raw, y_train, y_test, 
                                   feature_cols, k_list=[10, 25, 50, 100, 200]):
    """
    Feature subset sweep: preprocess -> importance ranking -> her k için model train.
    Returns: results_list (her k için metrikler + floor comparison)
    """
    # Preprocess
    X_tr, X_te, _ = preprocess_scenario(X_train_raw, X_test_raw)
    
    # XGBoost importance ranking
    importance_df = get_xgb_feature_importance_ranking(X_tr, y_train)
    
    # k_list + "all"
    k_list_full = list(k_list) + [None]  # None = all features
    
    results_list = []
    
    for k in k_list_full:
        if k is None:
            selected_features = list(X_tr.columns)
            k_label = 'all'
        else:
            k = min(k, len(importance_df))  # Handle k > n_features
            selected_features = importance_df.head(k)['feature'].tolist()
            k_label = f'{k}'
        
        X_tr_k = X_tr[selected_features].copy()
        X_te_k = X_te[selected_features].copy()
        
        # BalancedBaggingClassifier
        base_est = lgb.LGBMClassifier(
            n_estimators=100,
            learning_rate=0.1,
            num_leaves=31,
            random_state=SEED,
            verbose=-1
        )
        model = BalancedBaggingClassifier(
            estimator=base_est,
            n_estimators=10,
            sampling_strategy='auto',
            replacement=False,
            random_state=SEED,
            n_jobs=-1
        )
        model.fit(X_tr_k, y_train)
        
        # OOF probabilities for threshold tuning
        y_prob_train = model.predict_proba(X_tr_k)[:, 1]
        y_prob_test = model.predict_proba(X_te_k)[:, 1]
        
        # Threshold tuning (%80/20)
        thr_8020, thr_mcc = optimize_threshold_8020(y_train, y_prob_train)
        
        # Test evaluation
        y_pred_test = (y_prob_test >= thr_8020).astype(int)
        
        # Train metrics (gap control)
        y_pred_train = (y_prob_train >= thr_8020).astype(int)
        train_metrics = compute_all_metrics(y_train, y_pred_train, y_prob_train)
        
        # Test metrics (50/50 dağılım)
        test_metrics = compute_all_metrics(y_test, y_pred_test, y_prob_test)
        
        # Bootstrap %80/20 eval
        boot_eval = bootstrap_8020_eval(y_test, y_prob_test, thr_8020)
        
        # Floor comparison
        f1_8020 = boot_eval['f1_mean']
        vs_floor = f1_8020 - FLOOR_F1_THRESHOLD
        
        result = {
            'panel': panel_name,
            'k': k_label,
            'n_features': len(selected_features),
            'f1_8020': f1_8020,
            'f1_8020_std': boot_eval['f1_std'],
            'f1_8020_ci_lo': boot_eval['f1_ci_lo'],
            'f1_8020_ci_hi': boot_eval['f1_ci_hi'],
            'mcc_8020': boot_eval['mcc_mean'],
            'prec_8020': boot_eval['prec_mean'],
            'rec_8020': boot_eval['rec_mean'],
            'f1_5050': test_metrics['f1'],
            'f1_vs_floor': vs_floor,
            'floor': FLOOR_F1_THRESHOLD,
            'train_f1': train_metrics['f1'],
            'train_test_gap': train_metrics['f1'] - test_metrics['f1'],
            'thr_8020': thr_8020,
        }
        results_list.append(result)
        
        print(f'  [{panel_name}/k={k_label}] F1_8020={f1_8020:.4f} [{boot_eval["f1_ci_lo"]:.3f}-{boot_eval["f1_ci_hi"]:.3f}] vs_floor={vs_floor:+.4f}')
    
    return results_list, importance_df

print('Feature subset training fonksiyonu hazır.')

Feature subset training fonksiyonu hazır.


In [7]:
# Cell 7: MASTER Panel — Feature Selection Sweep
print('='*70)
print('MASTER Panel — Feature Selection EDA Validation')
print('='*70)

results_master, importance_master = train_and_eval_feature_subset(
    'MASTER', X_train_m, X_test_m, y_train_m, y_test_m, feat_m,
    k_list=[10, 25, 50, 100, 200]
)

results_df_master = pd.DataFrame(results_master)
print(f'\nMASTER tamamlandı. Sonuçlar:')
print(results_df_master[['panel', 'k', 'n_features', 'f1_8020', 'f1_8020_ci_lo', 'f1_8020_ci_hi', 'f1_vs_floor']])

MASTER Panel — Feature Selection EDA Validation


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [MASTER/k=10] F1_8020=0.4612 [0.355-0.555] vs_floor=+0.1279


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [MASTER/k=25] F1_8020=0.5624 [0.469-0.657] vs_floor=+0.2290


...

  [MASTER/k=100] F1_8020=0.5493 [0.450-0.635] vs_floor=+0.2160


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [MASTER/k=200] F1_8020=0.5678 [0.488-0.640] vs_floor=+0.2345


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [MASTER/k=all] F1_8020=0.5578 [0.472-0.624] vs_floor=+0.2244

MASTER tamamlandı. Sonuçlar:
    panel    k  n_features   f1_8020  f1_8020_ci_lo  f1_8020_ci_hi  \
0  MASTER   10          10  0.461185       0.355378       0.555423   
1  MASTER   25          25  0.562353       0.468978       0.657214   
2  MASTER   50          50  0.556556       0.459564       0.656250   
3  MASTER  100         100  0.549321       0.450123       0.635500   
4  MASTER  200         200  0.567833       0.488108       0.640160   
5  MASTER  all         426  0.557753       0.472378       0.623581   

   f1_vs_floor  
0     0.127852  
1     0.229020  
2     0.223222  
3     0.215988  
4     0.234500  
5     0.224420  


In [8]:
# Cell 8: KANSER Panel — Feature Selection Sweep
print('='*70)
print('KANSER Panel — Feature Selection EDA Validation')
print('='*70)

X_train_k, X_test_k, y_train_k, y_test_k, feat_k, const_k, dup_k = load_and_clean_panel('KANSER')

results_kanser, importance_kanser = train_and_eval_feature_subset(
    'KANSER', X_train_k, X_test_k, y_train_k, y_test_k, feat_k,
    k_list=[10, 25, 50, 100, 200]
)

results_df_kanser = pd.DataFrame(results_kanser)
print(f'\nKANSER tamamlandı. Sonuçlar:')
print(results_df_kanser[['panel', 'k', 'n_features', 'f1_8020', 'f1_8020_ci_lo', 'f1_8020_ci_hi', 'f1_vs_floor']])

KANSER Panel — Feature Selection EDA Validation
KANSER yüklendi: (388, 353)
Label dağılımı:
Label
1    268
0    120
Name: count, dtype: int64
Train: (310, 280) (pos=214, neg=96)
Test:  (78, 280) (pos=54, neg=24)
Sabit=69, Özdeş çift=694, Toplam drop=71, Kalan=280



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [KANSER/k=10] F1_8020=0.4377 [0.268-0.688] vs_floor=+0.1044


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [KANSER/k=25] F1_8020=0.5776 [0.343-0.740] vs_floor=+0.2443


...

  [KANSER/k=100] F1_8020=0.6319 [0.348-0.800] vs_floor=+0.2986


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [KANSER/k=200] F1_8020=0.5891 [0.328-0.757] vs_floor=+0.2558


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [KANSER/k=all] F1_8020=0.6262 [0.348-0.800] vs_floor=+0.2929

KANSER tamamlandı. Sonuçlar:
    panel    k  n_features   f1_8020  f1_8020_ci_lo  f1_8020_ci_hi  \
0  KANSER   10          10  0.437743       0.268030       0.687684   
1  KANSER   25          25  0.577591       0.342708       0.740074   
2  KANSER   50          50  0.565656       0.358333       0.697059   
3  KANSER  100         100  0.631919       0.348333       0.800000   
4  KANSER  200         200  0.589114       0.328462       0.756868   
5  KANSER  all         502  0.626214       0.348333       0.800000   

   f1_vs_floor  
0     0.104410  
1     0.244258  
2     0.232323  
3     0.298586  
4     0.255780  
5     0.292881  


In [9]:
# Cell 9: PAH Panel — Feature Selection Sweep
print('='*70)
print('PAH Panel — Feature Selection EDA Validation')
print('='*70)

X_train_p, X_test_p, y_train_p, y_test_p, feat_p, const_p, dup_p = load_and_clean_panel('PAH')

results_pah, importance_pah = train_and_eval_feature_subset(
    'PAH', X_train_p, X_test_p, y_train_p, y_test_p, feat_p,
    k_list=[10, 25, 50, 100, 200]
)

results_df_pah = pd.DataFrame(results_pah)
print(f'\nPAH tamamlandı. Sonuçlar:')
print(results_df_pah[['panel', 'k', 'n_features', 'f1_8020', 'f1_8020_ci_lo', 'f1_8020_ci_hi', 'f1_vs_floor']])

PAH Panel — Feature Selection EDA Validation
PAH yüklendi: (372, 353)
Label dağılımı:
Label
1    310
0     62
Name: count, dtype: int64
Train: (297, 258) (pos=248, neg=49)
Test:  (75, 258) (pos=62, neg=13)
Sabit=91, Özdeş çift=886, Toplam drop=93, Kalan=258



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [PAH/k=10] F1_8020=0.3024 [0.032-0.500] vs_floor=-0.0309


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [PAH/k=25] F1_8020=0.3226 [0.129-0.545] vs_floor=-0.0107


...

  [PAH/k=100] F1_8020=0.3104 [0.143-0.491] vs_floor=-0.0229


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [PAH/k=200] F1_8020=0.3101 [0.143-0.500] vs_floor=-0.0232


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  [PAH/k=all] F1_8020=0.3101 [0.143-0.500] vs_floor=-0.0232

PAH tamamlandı. Sonuçlar:
  panel    k  n_features   f1_8020  f1_8020_ci_lo  f1_8020_ci_hi  f1_vs_floor
0   PAH   10          10  0.302420       0.032143       0.500000    -0.030913
1   PAH   25          25  0.322605       0.129018       0.545455    -0.010729
2   PAH   50          50  0.367745       0.185417       0.565584     0.034411
3   PAH  100         100  0.310396       0.142857       0.491346    -0.022938
4   PAH  200         200  0.310113       0.142857       0.500000    -0.023220
5   PAH  all         408  0.310113       0.142857       0.500000    -0.023220


In [10]:
# Cell 10: Buyuk Karsilastirma Tablosu (tum paneller)
print('='*90)
print('KAPSAMLI KARŞILAŞTIRMA TABLOSU (MASTER, KANSER, PAH)')
print('='*90)

all_results = pd.concat([results_df_master, results_df_kanser, results_df_pah], ignore_index=True)

# Summary table
summary_cols = ['panel', 'k', 'n_features', 'f1_8020', 'f1_8020_ci_lo', 'f1_8020_ci_hi', 
                'mcc_8020', 'prec_8020', 'rec_8020', 'f1_vs_floor']
summary_df = all_results[summary_cols].copy()
summary_df['ci'] = (summary_df['f1_8020_ci_lo'].astype(str).str[:6] + 
                    '-' + summary_df['f1_8020_ci_hi'].astype(str).str[:6])

print(summary_df.to_string(index=False))

# Save to CSV
csv_path = os.path.join(RESULTS_DIR, 'nb41_results.csv')
all_results.to_csv(csv_path, index=False)
print(f'\nCSV kaydedildi: {csv_path}')

# Best k per panel
print('\n' + '='*90)
print('EN İYİ k (F1_8020 maksimum):')
print('='*90)
for panel in ['MASTER', 'KANSER', 'PAH']:
    panel_data = all_results[all_results['panel'] == panel]
    best_row = panel_data.loc[panel_data['f1_8020'].idxmax()]
    all_row = panel_data[panel_data['k'] == 'all'].iloc[0] if len(panel_data[panel_data['k'] == 'all']) > 0 else None
    
    print(f'\n{panel}:')
    print(f'  Best k={best_row["k"]}: F1_8020={best_row["f1_8020"]:.4f} [{best_row["f1_8020_ci_lo"]:.3f}-{best_row["f1_8020_ci_hi"]:.3f}]')
    if all_row is not None:
        diff = best_row['f1_8020'] - all_row['f1_8020']
        print(f'  All features: F1_8020={all_row["f1_8020"]:.4f} [{all_row["f1_8020_ci_lo"]:.3f}-{all_row["f1_8020_ci_hi"]:.3f}]')
        print(f'  Fark: {diff:+.4f} (top-k advantage)')

KAPSAMLI KARŞILAŞTIRMA TABLOSU (MASTER, KANSER, PAH)
 panel   k  n_features  f1_8020  f1_8020_ci_lo  f1_8020_ci_hi  mcc_8020  prec_8020  rec_8020  f1_vs_floor            ci
MASTER  10          10 0.461185       0.355378       0.555423  0.306587   0.397302  0.555385     0.127852 0.3553-0.5554
MASTER  25          25 0.562353       0.468978       0.657214  0.442280   0.459601  0.728718     0.229020 0.4689-0.6572
MASTER  50          50 0.556556       0.459564       0.656250  0.433708   0.462824  0.702051     0.223222 0.4595-0.6562
MASTER 100         100 0.549321       0.450123       0.635500  0.426648   0.433718  0.753846     0.215988 0.4501-0.6355
MASTER 200         200 0.567833       0.488108       0.640160  0.454844   0.442239  0.796923     0.234500 0.4881-0.6401
MASTER all         426 0.557753       0.472378       0.623581  0.441107   0.432454  0.789744     0.224420 0.4723-0.6235
KANSER  10          10 0.437743       0.268030       0.687684  0.260061   0.312730  0.750000     0.104410 0

In [11]:
# Cell 11: Gorsellestirilmeler

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- Fig 1: k-sweep F1_8020 egrileri (panel başına) ---
ax = axes[0, 0]
for panel in ['MASTER', 'KANSER', 'PAH']:
    panel_data = all_results[all_results['panel'] == panel].sort_values('n_features')
    # Non-all features only for curve
    curve_data = panel_data[panel_data['k'] != 'all']
    if len(curve_data) > 0:
        ax.plot(curve_data['n_features'], curve_data['f1_8020'], 'o-', label=panel, markersize=7)

ax.axhline(y=FLOOR_F1_THRESHOLD, color='red', linestyle='--', alpha=0.5, label=f'Floor (prev=0.20)')
ax.set_xlabel('Number of Features')
ax.set_ylabel('F1_8020 (Bootstrap Mean)')
ax.set_title('Feature Sweep: k vs F1_8020')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Fig 2: top-k vs all comparison bar ---
ax = axes[0, 1]
comparison_data = []
for panel in ['MASTER', 'KANSER', 'PAH']:
    panel_data = all_results[all_results['panel'] == panel]
    best_k = panel_data[panel_data['k'] != 'all'].nlargest(1, 'f1_8020')
    all_k = panel_data[panel_data['k'] == 'all']
    if len(best_k) > 0 and len(all_k) > 0:
        comparison_data.append({
            'Panel': panel,
            'Best-k': best_k.iloc[0]['f1_8020'],
            'All': all_k.iloc[0]['f1_8020']
        })
comparison_df_vis = pd.DataFrame(comparison_data)
comparison_df_vis.set_index('Panel')[['Best-k', 'All']].plot(kind='bar', ax=ax)
ax.set_title('Top-k vs All Features (F1_8020)')
ax.set_ylabel('F1_8020')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.grid(True, alpha=0.3, axis='y')

# --- Fig 3: Precision-Recall scatter per panel ---
ax = axes[0, 2]
for panel in ['MASTER', 'KANSER', 'PAH']:
    panel_data = all_results[all_results['panel'] == panel]
    ax.scatter(panel_data['prec_8020'], panel_data['rec_8020'], 
              label=panel, s=100, alpha=0.6)
ax.set_xlabel('Precision_8020')
ax.set_ylabel('Recall_8020')
ax.set_title('Precision-Recall Trade-off')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Fig 4: Feature importance top-20 (MASTER) ---
ax = axes[1, 0]
importance_master_top20 = importance_master.head(20)
ax.barh(range(len(importance_master_top20)), importance_master_top20['importance'].values)
ax.set_yticks(range(len(importance_master_top20)))
ax.set_yticklabels(importance_master_top20['feature'].values, fontsize=8)
ax.set_xlabel('XGBoost Importance (Gain)')
ax.set_title('MASTER: Top-20 Features by Importance')
ax.invert_yaxis()

# --- Fig 5: Feature importance top-20 (KANSER) ---
ax = axes[1, 1]
importance_kanser_top20 = importance_kanser.head(20)
ax.barh(range(len(importance_kanser_top20)), importance_kanser_top20['importance'].values)
ax.set_yticks(range(len(importance_kanser_top20)))
ax.set_yticklabels(importance_kanser_top20['feature'].values, fontsize=8)
ax.set_xlabel('XGBoost Importance (Gain)')
ax.set_title('KANSER: Top-20 Features by Importance')
ax.invert_yaxis()

# --- Fig 6: Feature importance top-20 (PAH) ---
ax = axes[1, 2]
importance_pah_top20 = importance_pah.head(20)
ax.barh(range(len(importance_pah_top20)), importance_pah_top20['importance'].values)
ax.set_yticks(range(len(importance_pah_top20)))
ax.set_yticklabels(importance_pah_top20['feature'].values, fontsize=8)
ax.set_xlabel('XGBoost Importance (Gain)')
ax.set_title('PAH: Top-20 Features by Importance')
ax.invert_yaxis()

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'fig1_ksweep_and_importance.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Figure kaydedildi: {fig_path}')
plt.close()

Figure kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v24_feature_selection/fig1_ksweep_and_importance.png


In [12]:
# Cell 12: Overfit Analizi — Train-Test Gap

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, panel in enumerate(['MASTER', 'KANSER', 'PAH']):
    ax = axes[i]
    panel_data = all_results[all_results['panel'] == panel]
    
    x = range(len(panel_data))
    ax.plot(x, panel_data['train_f1'], 'o-', label='Train F1', markersize=6)
    ax.plot(x, panel_data['f1_5050'], 's-', label='Test F1 (50/50)', markersize=6)
    ax.plot(x, panel_data['f1_8020'], '^-', label='Test F1 (80/20)', markersize=6)
    
    ax.set_xticks(x)
    ax.set_xticklabels(panel_data['k'].values, rotation=45)
    ax.set_xlabel('Feature Subset (k)')
    ax.set_ylabel('F1 Score')
    ax.set_title(f'{panel}: Train-Test Gap')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'fig2_train_test_gap.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Figure kaydedildi: {fig_path}')
plt.close()

Figure kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v24_feature_selection/fig2_train_test_gap.png


In [13]:
# Cell 13: Floor Comparison — Each Result vs Trivial Baseline

print('\n' + '='*90)
print('FLOOR KARSILASTIRMASI (trivial baseline prev=0.20)')
print('='*90)

floor_check = all_results[['panel', 'k', 'f1_8020', 'f1_vs_floor', 'floor']].copy()
floor_check['passes_floor'] = floor_check['f1_vs_floor'] > 0
floor_check['margin'] = (floor_check['f1_8020'] / floor_check['floor']).round(2)

print(floor_check.to_string(index=False))

# Summary
n_pass = (floor_check['passes_floor']).sum()
n_total = len(floor_check)
print(f'\nPasses floor ({FLOOR_F1_THRESHOLD:.3f}): {n_pass}/{n_total} ({100*n_pass/n_total:.1f}%)')

# Panel-wise summary
print('\nPanel-wise floor passage:')
for panel in ['MASTER', 'KANSER', 'PAH']:
    panel_check = floor_check[floor_check['panel'] == panel]
    pass_rate = (panel_check['passes_floor']).mean()
    print(f'  {panel}: {pass_rate:.1%}')


FLOOR KARSILASTIRMASI (trivial baseline prev=0.20)
 panel   k  f1_8020  f1_vs_floor    floor  passes_floor  margin
MASTER  10 0.461185     0.127852 0.333333          True    1.38
MASTER  25 0.562353     0.229020 0.333333          True    1.69
MASTER  50 0.556556     0.223222 0.333333          True    1.67
MASTER 100 0.549321     0.215988 0.333333          True    1.65
MASTER 200 0.567833     0.234500 0.333333          True    1.70
MASTER all 0.557753     0.224420 0.333333          True    1.67
KANSER  10 0.437743     0.104410 0.333333          True    1.31
KANSER  25 0.577591     0.244258 0.333333          True    1.73
KANSER  50 0.565656     0.232323 0.333333          True    1.70
KANSER 100 0.631919     0.298586 0.333333          True    1.90
KANSER 200 0.589114     0.255780 0.333333          True    1.77
KANSER all 0.626214     0.292881 0.333333          True    1.88
   PAH  10 0.302420    -0.030913 0.333333         False    0.91
   PAH  25 0.322605    -0.010729 0.333333         Fa

In [14]:
# Cell 14: Ana Hipotez Testi — Danisan EDA Bulgusu Geçerli mi?

print('\n' + '='*90)
print('ANA HİPOTEZ TESTİ: Danışman EDA (top-200 > all) bizim protokolümüzde de geçerli mi?')
print('='*90)

for panel in ['MASTER', 'KANSER', 'PAH']:
    panel_data = all_results[all_results['panel'] == panel]
    
    # En iyi k vs all
    best_k_row = panel_data[panel_data['k'] != 'all'].nlargest(1, 'f1_8020').iloc[0]
    all_row = panel_data[panel_data['k'] == 'all']
    
    print(f'\n{panel}:')
    print(f'  Best top-k (k={best_k_row["k"]}):')
    print(f'    F1_8020 = {best_k_row["f1_8020"]:.4f} [{best_k_row["f1_8020_ci_lo"]:.3f}-{best_k_row["f1_8020_ci_hi"]:.3f}]')
    print(f'    n_features = {int(best_k_row["n_features"])}')
    
    if len(all_row) > 0:
        all_val = all_row.iloc[0]
        print(f'  All features:')
        print(f'    F1_8020 = {all_val["f1_8020"]:.4f} [{all_val["f1_8020_ci_lo"]:.3f}-{all_val["f1_8020_ci_hi"]:.3f}]')
        print(f'    n_features = {int(all_val["n_features"])}')
        
        diff = best_k_row['f1_8020'] - all_val['f1_8020']
        ci_overlap = not (best_k_row['f1_8020_ci_lo'] > all_val['f1_8020_ci_hi'] or 
                          all_val['f1_8020_ci_lo'] > best_k_row['f1_8020_ci_hi'])
        
        print(f'  Fark: {diff:+.4f}')
        if diff > 0.010:
            print(f'  ✓ TOP-K KAZANDI (güçlü): {diff:.4f} iyileştirme')
        elif diff > 0.002 and not ci_overlap:
            print(f'  ≈ TOP-K HAFİF KAZANDI (marjinal ama CI haricinde)')
        elif not ci_overlap:
            print(f'  ✗ ALL FEATURES KAZANDI: {-diff:.4f}')
        else:
            print(f'  ~ CI OVERLAP: Fark istatistiksel olarak anlamlı değil')


ANA HİPOTEZ TESTİ: Danışman EDA (top-200 > all) bizim protokolümüzde de geçerli mi?

MASTER:
  Best top-k (k=200):
    F1_8020 = 0.5678 [0.488-0.640]
    n_features = 200
  All features:
    F1_8020 = 0.5578 [0.472-0.624]
    n_features = 426
  Fark: +0.0101
  ✓ TOP-K KAZANDI (güçlü): 0.0101 iyileştirme

KANSER:
  Best top-k (k=100):
    F1_8020 = 0.6319 [0.348-0.800]
    n_features = 100
  All features:
    F1_8020 = 0.6262 [0.348-0.800]
    n_features = 502
  Fark: +0.0057
  ~ CI OVERLAP: Fark istatistiksel olarak anlamlı değil

PAH:
  Best top-k (k=50):
    F1_8020 = 0.3677 [0.185-0.566]
    n_features = 50
  All features:
    F1_8020 = 0.3101 [0.143-0.500]
    n_features = 408
  Fark: +0.0576
  ✓ TOP-K KAZANDI (güçlü): 0.0576 iyileştirme


In [15]:
# Cell 15: PDF Rapor Olustur

class NB41Report(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.cell(0, 10, 'NB41: Feature Selection EDA Validation', 0, 1, 'C')
        self.set_font('Helvetica', '', 9)
        self.cell(0, 5, f'SEED={SEED} | TEST_SIZE={TEST_SIZE} | 3 panel x k-sweep', 0, 1, 'C')
        self.ln(3)

    def section_title(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.set_fill_color(41, 128, 185)
        self.set_text_color(255, 255, 255)
        self.cell(0, 8, f'  {title}', 0, 1, 'L', fill=True)
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def body_text(self, text):
        self.set_font('Helvetica', '', 9)
        self.multi_cell(0, 5, text)
        self.ln(2)

    def add_table(self, headers, rows, col_widths=None):
        if col_widths is None:
            col_widths = [self.epw / len(headers)] * len(headers)
        self.set_font('Helvetica', 'B', 7)
        self.set_fill_color(52, 73, 94)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, str(h), 1, 0, 'C', fill=True)
        self.ln()
        self.set_font('Helvetica', '', 6.5)
        self.set_text_color(0, 0, 0)
        for j, row in enumerate(rows):
            if j % 2 == 0:
                self.set_fill_color(236, 240, 241)
            else:
                self.set_fill_color(255, 255, 255)
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, 'C', fill=True)
            self.ln()
        self.ln(3)


pdf = NB41Report()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# --- 0. Hipotez ---
pdf.section_title('0. Hipotez ve Baglam')
pdf.body_text(
    'Danisman EDA: XGBoost feature importance ile MASTER icinde top-200 feature, '
    'tum feature lerden daha iyi (train-dagilimi F1: 0.894 vs 0.879). '
    'Kuyruk feature lar gurultu mu?\n'
    'Bizim soru: Bu kazanc, reverse dagilim (%80/20 bootstrap) protokolumuzde '
    'de gecerli mi? 3 panelde test ettik.'
)

# --- 1. Deney ---
pdf.section_title('1. Deney Tasarimi')
pdf.body_text(
    f'Her panel icin:\n'
    f'- Stratified %80/20 split (SEED={SEED})\n'
    f'- M3 missing (is_missing + medyan imputation)\n'
    f'- XGBoost importance ranking (gain)\n'
    f'- Feature subset sweep: k in [10, 25, 50, 100, 200, all]\n'
    f'- BalancedBaggingClassifier (10x LGBM)\n'
    f'- Eval: NB39 protokol (%80/20 bootstrap, N={N_BOOTSTRAP}, 95% CI)\n'
    f'- Floor reference: F1_floor = {FLOOR_F1_THRESHOLD:.3f} (prev=0.20)'
)

# --- 2. Sonuclar (her panel) ---
for panel in ['MASTER', 'KANSER', 'PAH']:
    pdf.section_title(f'2. {panel} Sonuclari')
    panel_data = all_results[all_results['panel'] == panel]

    headers = ['k', 'n_feat', 'F1_8020', 'CI_95', 'MCC', 'Prec', 'Rec', 'vs_floor']
    rows = []
    for _, row in panel_data.iterrows():
        ci = f"[{row['f1_8020_ci_lo']:.3f}-{row['f1_8020_ci_hi']:.3f}]"
        rows.append([
            str(row['k']), str(int(row['n_features'])),
            f"{row['f1_8020']:.4f}", ci,
            f"{row['mcc_8020']:.3f}", f"{row['prec_8020']:.3f}",
            f"{row['rec_8020']:.3f}", f"{row['f1_vs_floor']:+.3f}"
        ])
    pdf.add_table(headers, rows, [15, 15, 18, 30, 15, 15, 15, 18])

# --- 3. Ana hipotez testi ---
pdf.add_page()
pdf.section_title('3. Ana Hipotez Testi: Top-k vs All')

hyp_rows = []
for panel in ['MASTER', 'KANSER', 'PAH']:
    panel_data = all_results[all_results['panel'] == panel]
    best_k = panel_data[panel_data['k'] != 'all'].nlargest(1, 'f1_8020').iloc[0]
    all_k = panel_data[panel_data['k'] == 'all'].iloc[0] if len(panel_data[panel_data['k'] == 'all']) > 0 else None

    if all_k is not None:
        diff = best_k['f1_8020'] - all_k['f1_8020']
        hyp_rows.append([
            panel,
            f"k={best_k['k']}", f"{best_k['f1_8020']:.4f}",
            'All', f"{all_k['f1_8020']:.4f}",
            f"{diff:+.4f}"
        ])

pdf.add_table(
    ['Panel', 'Best', 'F1', 'Comp', 'F1', 'Diff'],
    hyp_rows,
    [25, 20, 20, 20, 20, 20]
)

pdf.body_text(
    'SONUC: Danisman EDA bulgusu (top-200 > all) MASTER/KANSER/PAH icinde '
    'reverse dagilim protokolumuzde de gecerli mi?\n\n'
    '- MASTER: Danisman top-200 onerisini test ettik.\n'
    '- KANSER: Kucuk panel, top-50/100 avantaji olabilir.\n'
    '- PAH: Guclu dengesizlik (%17 pos), top-k daha avantajli olabilir.\n\n'
    'Floor karsilastirmasi: vs_floor sutununa bak; pozitif ise model '
    'trivial baseline i geciyor.'
)

# Gorseller ekle
for img_name in ['fig1_ksweep_and_importance.png', 'fig2_train_test_gap.png']:
    img_path = os.path.join(RESULTS_DIR, img_name)
    if os.path.exists(img_path):
        pdf.add_page()
        pdf.section_title(f'Gorsel: {img_name}')
        try:
            pdf.image(img_path, x=10, w=190)
        except Exception:
            pdf.body_text(f'[Gorsel yuklenemedi: {img_name}]')

# Kaydet
report_path = os.path.join(REPORTS_DIR, 'NB41_feature_selection_report.pdf')
pdf.output(report_path)
print(f'PDF rapor kaydedildi: {report_path}')

PDF rapor kaydedildi: /Users/tefe/teknofest_model/teknofest_model/reports/NB41_feature_selection_report.pdf


In [16]:
# Cell 16: Ozet ve Bulguler

print('\n' + '='*90)
print('NB41 TAMAMLANDI — FEATURE SELECTION EDA VALIDATION')
print('='*90)

print(f'\nÇıktılar:')
print(f'  CSV: {RESULTS_DIR}/nb41_results.csv')
print(f'  PNG1: {RESULTS_DIR}/fig1_ksweep_and_importance.png')
print(f'  PNG2: {RESULTS_DIR}/fig2_train_test_gap.png')
print(f'  PDF: {REPORTS_DIR}/NB41_feature_selection_report.pdf')

print(f'\nHipotez Yanıt:')
print(f'  Danişan EDA (top-200 > all) reverse-dağılım protokolümüzde de geçerli mi?')
print(f'  -> Tabloları kontrol et ve sonuçları yorumla.')

print(f'\nKay detaylar:')
print(f'  - Floor referansı: F1={FLOOR_F1_THRESHOLD:.3f} (trivial baseline, prev=0.20)')
print(f'  - %80/20 Bootstrap: N={N_BOOTSTRAP} sample, %95 CI')
print(f'  - Paneller: MASTER (n=2931), KANSER (n=388), PAH (n=372)')
print(f'  - CFTR atlanmış (n=21 benign, CI=[0–1] anlamsız)')


NB41 TAMAMLANDI — FEATURE SELECTION EDA VALIDATION

Çıktılar:
  CSV: /Users/tefe/teknofest_model/teknofest_model/results/v24_feature_selection/nb41_results.csv
  PNG1: /Users/tefe/teknofest_model/teknofest_model/results/v24_feature_selection/fig1_ksweep_and_importance.png
  PNG2: /Users/tefe/teknofest_model/teknofest_model/results/v24_feature_selection/fig2_train_test_gap.png
  PDF: /Users/tefe/teknofest_model/teknofest_model/reports/NB41_feature_selection_report.pdf

Hipotez Yanıt:
  Danişan EDA (top-200 > all) reverse-dağılım protokolümüzde de geçerli mi?
  -> Tabloları kontrol et ve sonuçları yorumla.

Kay detaylar:
  - Floor referansı: F1=0.333 (trivial baseline, prev=0.20)
  - %80/20 Bootstrap: N=50 sample, %95 CI
  - Paneller: MASTER (n=2931), KANSER (n=388), PAH (n=372)
  - CFTR atlanmış (n=21 benign, CI=[0–1] anlamsız)
